# Land valuation

This notebook produces a per-parcel land-value layer separately from full market value, then scores it against the Lars-Tests (the LVT-incentive-aligned tests, distinct from IAAO ratio studies).

What it does:

1. Loads the cleaned `SalesUniversePair` (from `02-clean.ipynb`) plus an ensemble market-value prediction (from `03-model.ipynb`).
2. Builds a de-facto `(jurisdiction, zoning)` reference table from the parcel data and joins minimum-lot-size lookups onto every parcel.
3. Builds the neighborhood-cascade hierarchy walked when a finest-grained cell is evidence-thin.
4. Curates the witness pool — six streams (W1 vacant, W2 teardown, W3 extraction, W4 low-FAR, W5 prior-xfer, W6 pred-residual) with contamination filters.
5. Paints every parcel with the production table painter — per-cell `base_lot` / `size_curve` / `puv` rules, with size-curve breakpoints fit empirically (in zoning-min units) from the witness pool.
6. Scores the painted output on the L1-L6 Lars-Tests, side-by-side with the assessor's own `assr_land_value` as a baseline.

Locality-specific knobs (cascade column names, multiplicative adjustment factors) are picked up automatically from `data/<locality>/in/land_config.py` if present. Without that file the notebook falls back to the `neighborhood` column alone with no adjustments — useful for a first pass on a new locality.

If your jurisdiction's neighborhood codes encode a positional substructure (e.g. Wake's 7-char VCS `[area:2][juris:2][sub:3]`), declare the prefix-derived cascade columns in `data.load.<id>.calc` in `settings.json` using the `substr` operator. They're then populated during the assemble step rather than re-derived here. Example:

```json
"vcs_area_juris": ["substr", "neighborhood", {"left": 0, "right": 4}]
```

Pre-requisites:
* Notebook 2 must have produced `out/2-clean-sup.pickle`.
* Notebook 3 must have produced an ensemble prediction at `out/models/<model_group>/main/ensemble/pred_universe.parquet`.

**User settings**

Change these as desired:

In [ ]:
# The slug of the locality you are currently working on
locality = "us-nc-wake"

# Which model group to value land for. v1 supports single_family.
model_group = "single_family"

# Valuation date — typically the assessment-effective date for your jurisdiction.
valuation_date = "2024-01-01"

# Output subfolder (under data/<locality>/out/land/) for this run's artifacts.
out_subfolder = "rung1.5"

# Whether to print out a lot of stuff (can help with debugging) or stay mostly quiet
verbose = True

# 1. Basic setup

In [ ]:
import init_notebooks
init_notebooks.setup_environment()
locality = init_notebooks.check_for_different_locality(locality)

In [ ]:
# Pipeline wrappers (canonical entry points for the land flow)
from openavmkit.pipeline import (
    init_notebook,
    load_settings,
    read_pickle,
    enrich_with_empirical_zoning,
    build_land_hierarchy,
    curate_land_witnesses,
    run_land_painter_tables,
    run_land_lars_tests,
)
from openavmkit.land import (
    WitnessConfig,
    write_land_tables_artifact,
    write_evidence_packets,
)

import os
import sys
import numpy as np
import pandas as pd

In [ ]:
init_notebook(locality)
settings = load_settings()
valuation_date = pd.Timestamp(valuation_date)

In [ ]:
# Optional locality-specific land config. Lives in data/<locality>/in/land_config.py.
# Provides the cascade column list and adjustment factors that encode jurisdiction-
# specific assessor-coding conventions. Without this file the notebook falls back to
# `[neighborhood]` alone with no multiplicative adjustments.
_in_dir = os.path.abspath("in")
if _in_dir not in sys.path:
    sys.path.insert(0, _in_dir)
try:
    from land_config import cascade_levels, default_adjustments  # type: ignore
    levels = cascade_levels()
    adjustments = default_adjustments()
    print(f"Loaded locality config from {_in_dir}/land_config.py")
    print(f"  cascade levels: {levels}")
    print(f"  adjustments:    {len(adjustments)} factors")
except (ImportError, AttributeError) as exc:
    levels = ["neighborhood"]
    adjustments = None
    print(f"No in/land_config.py found ({exc}); falling back to {levels} with no adjustments.")

# 2. Load cleaned SUP + ensemble prediction

The painter calibrates each cell's land-value total against the cost-residual `prediction - assr_impr_value` from the ensemble model. Both the cleaned universe and the ensemble prediction must already exist on disk.

In [ ]:
sup = read_pickle("out/2-clean-sup")
print(f"universe: {sup.universe.shape}")
print(f"sales:    {sup.sales.shape}")

universe = sup.universe.copy()
if "floor_area_ratio" not in universe.columns:
    universe["floor_area_ratio"] = np.where(
        universe["land_area_sqft"] > 0,
        universe["bldg_area_finished_sqft"] / universe["land_area_sqft"],
        np.nan,
    )

pred_path = f"out/models/{model_group}/main/ensemble/pred_universe.parquet"
ens = pd.read_parquet(pred_path)[["key", "prediction"]]
universe = universe.merge(ens, on="key", how="left")
print(f"merged ensemble prediction: notnull={universe['prediction'].notna().sum():,}")

# 3. Empirical zoning + neighborhood hierarchy

Empirical zoning derives a *de facto* `(jurisdiction, zoning)` reference table — minimum lot size at the 10th percentile of built parcels, FAR ceiling at the 90th — and joins those lookups onto every parcel. They feed the witness contamination filters (drop sales below local min lot) and the painter's zoning-anchored size-curve breakpoints.

The neighborhood hierarchy is the cascade walked when a finest-grained cell has too few witnesses. Levels are finest-first; the painter falls back from the finest-grained `neighborhood` column upward through coarser administrative levels until it finds a cell with enough witnesses.

All cascade columns must already exist on the universe — derive substring-prefix columns via the `substr` calc operator in `settings.json` (see the intro cell), and declare any administrative columns (`planning_jurisdiction`, `township`, etc.) in `data.load`. `build_land_hierarchy` does not derive columns; it just validates and assembles the spec.

In [ ]:
universe, zoning_table = enrich_with_empirical_zoning(universe, verbose=verbose)

In [ ]:
universe, spec = build_land_hierarchy(
    universe,
    levels=levels,
    verbose=verbose,
)

# 4. Restrict to the configured model group

The land flow is currently calibrated for one model group at a time. Single-family is the typical first target — its sales are dense enough to support both witness curation and the cost-residual calibration anchor.

In [ ]:
mg_universe = universe[universe["model_group"] == model_group].copy()
print(f"{model_group}: {len(mg_universe):,} parcels")

# Sales typically don't carry parcel-side neighborhood / zoning_min — merge from
# the universe so the witness contamination filters can group sales correctly.
sales = sup.sales.copy()
if "floor_area_ratio" not in sales.columns:
    sales["floor_area_ratio"] = np.where(
        sales["land_area_sqft"] > 0,
        sales["bldg_area_finished_sqft"] / sales["land_area_sqft"],
        np.nan,
    )
needed_from_universe = [
    c for c in [
        "neighborhood", "vcs_area_juris", "planning_jurisdiction", "township",
        "__county__", "model_group",
        "land_deferred_code", "historic_deferred_code",
        "zoning_emp_min_lot_sqft",
    ] if c in universe.columns
]
drop_existing = [c for c in needed_from_universe if c in sales.columns]
if drop_existing:
    sales = sales.drop(columns=drop_existing)
sales = sales.merge(
    universe[["key"] + needed_from_universe].drop_duplicates("key"),
    on="key",
    how="left",
)
mg_sales = sales[sales["model_group"] == model_group].copy()
print(f"{model_group} sales: {len(mg_sales):,}")

# 5. Witness pool

Six witness streams are pooled with a contamination filter:

* **W1** — clean vacant sales (gold for buildable sites)
* **W2** — teardown sales (vacant land beneath a soon-to-be-demolished structure)
* **W3** — extraction (new-construction sale price minus depreciated cost basis)
* **W4** — low-FAR sales (old small house on a big lot — sale price implies the land)
* **W5** — prior-xfer (parcel CSV's most recent vacant-land transfer)
* **W6** — pred-residual (full-universe `prediction - assr_impr_value`)

Anomaly filters drop sales that violate physical sanity (e.g., painted land + cost basis far below construction cost) or that are inconsistent with prior arms-length transfers.

In [ ]:
flagged_out: list = []
witnesses = curate_land_witnesses(
    mg_sales,
    mg_universe,
    cfg=WitnessConfig(),
    valuation_date=valuation_date,
    flagged_out=flagged_out,
    verbose=verbose,
)

# Wire cascade columns onto witnesses so the painter can group them at any level.
cascade_cols = [c for c in spec.levels if c in mg_universe.columns]
if "key" in mg_universe.columns:
    join_w = mg_universe[["key"] + cascade_cols].rename(columns={"key": "parcel_key"})
    overlap = [c for c in cascade_cols if c in witnesses.columns]
    if overlap:
        witnesses = witnesses.drop(columns=overlap)
    witnesses = witnesses.merge(join_w, on="parcel_key", how="left")

print(f"witness pool: {len(witnesses):,}")
print(witnesses["witness_kind"].value_counts().to_string())

# 6. Paint per-parcel land values

The table painter:

1. Empirically fits `(bp1_mult, bp2_mult)` for the size-curve from direct-evidence witnesses (W1/W3/W4/W5) — bin medians of cell-anchored normalized $/sqft are matched against the implied 3-tier average curve under fixed decays `(1.0, 0.5, 0.25)`; grid search picks the breakpoint pair (in `zoning_emp_min` units) that minimizes bin-weighted SSE.
2. Builds one `LandTable` per cascade cell — `base_lot` (uniform $/lot when within-cell lot-size CV is small), `size_curve` (zoning-anchored 3-tier marginal $/sqft), or `puv` (state-set Present-Use-Value rate).
3. Calibrates each cell's tier-1 rate so the painted total over built parcels equals the cost-residual total (`prediction - assr_impr_value`).
4. Walks the cascade and paints every parcel.
5. Applies the multiplicative adjustments (PUV, historic, partial-utilities, etc.) from `in/land_config.py`.

In [ ]:
painted, tables, fit_diag = run_land_painter_tables(
    mg_universe,
    witnesses,
    spec=spec,
    adjustments=adjustments,
    prediction_col="prediction",
    impr_value_col="assr_impr_value",
    valuation_date=valuation_date,
    verbose=verbose,
)
print(f"\npainted {(painted['land_value'] > 0).sum():,}/{len(painted):,} parcels with positive land value")
print(f"approach distribution:\n{painted['table_approach'].value_counts().to_string()}")

# 7. Score with Lars-Tests

The Lars-Tests (L1-L6) check LVT-incentive-aligned properties:

* **L1** improvement-neutrality — matched pairs (one improved, one cheap/vacant) should agree on land value
* **L2** within-cluster uniformity — side-by-side similar parcels should pay the same $/sqft (lower COD = better)
* **L3** vacant-burden flip — under a revenue-neutral LVT, vacant parcels should pay MORE than under property tax
* **L4** desirability — Spearman correlation between land $/sqft and total $/sqft (assr + predicted)
* **L5** consistent-use violators — parcels whose land value disagrees with their improvement use
* **L6** excess/surplus signal — large-lot parcels should reflect the excess-vs-surplus distinction

We score our painted output side-by-side with the assessor's own `assr_land_value` as a baseline.

In [ ]:
out_dir = os.path.join("out", "land", out_subfolder)
os.makedirs(out_dir, exist_ok=True)

lars_results = run_land_lars_tests(
    painted,
    candidates=[
        (f"{out_subfolder} painted", "land_value"),
        ("assr_land_value (baseline)", "assr_land_value"),
    ],
    out_path=os.path.join(out_dir, "lars_tests.md"),
    verbose=verbose,
)
print(f"\nLars-Tests report: {out_dir}/lars_tests.md")

# 8. Write artifacts

Three artifacts per run, all in `out/land/<out_subfolder>/`:

* `land_universe.parquet` / `.csv` — per-parcel painted land value, `$/sqft`, table used, approach, adjustment factors
* `land_tables.parquet` — one row per cascade cell with the table rule applied (base_lot value / tier breakpoints+rates / puv rate)
* `evidence_packets.parquet` — long table of `(neighborhood, witness, $/sqft)` so any per-parcel value can be traced back to its supporting market evidence

In [ ]:
out_cols = [c for c in [
    "key", "neighborhood", "vcs_area_juris", "planning_jurisdiction", "township",
    "land_area_sqft", "bldg_area_finished_sqft",
    "assr_land_value", "assr_impr_value", "assr_market_value",
    "prediction",
    "land_value", "land_value_per_sqft",
    "land_value_adjustment_factor", "land_value_adjustments_applied",
    "table_used", "table_approach",
    "zoning_emp_min_lot_sqft",
] if c in painted.columns]
painted[out_cols].to_parquet(os.path.join(out_dir, "land_universe.parquet"))
painted[out_cols].to_csv(os.path.join(out_dir, "land_universe.csv"), index=False)
print(f"land_universe.parquet ({len(painted):,} parcels)")

write_land_tables_artifact(tables, os.path.join(out_dir, "land_tables.parquet"))
print(f"land_tables.parquet ({len(tables):,} rows)")

write_evidence_packets(witnesses, os.path.join(out_dir, "evidence_packets.parquet"))
print(f"evidence_packets.parquet ({len(witnesses):,} witnesses)")

print(f"\nDone. Artifacts in {out_dir}/")